# Text Classification: HuggingFace Encoder + Fully Connected Head (PyTorch)

This notebook builds a text classifier from scratch using a **pretrained HuggingFace encoder** (DistilBERT) as a feature extractor, with a **custom fully connected (linear) classification head** on top, trained with plain PyTorch.

**Workflow:**
1. Get the data (HuggingFace `datasets`)
2. Randomly profile the data (peek at samples, check label balance)
3. Tokenize and wrap into a PyTorch `Dataset`
4. Put it into a `DataLoader` (batch size = 32)
5. Build the model (encoder + FC head)
6. Define loss (CrossEntropyLoss) and optimizer (Adam)
7. Train the model
8. Evaluate the model
9. Run inference on new text

Dataset used: **AG News** (4-class news topic classification: World / Sports / Business / Sci-Tech) — it's small, fast to download, and a clean example of multi-class text classification. Swap it out for your own dataset later; the rest of the pipeline doesn't need to change.

## 0. Install & Import Libraries

Uncomment the pip install line if you're running this in a fresh environment (e.g. Colab).

In [27]:
# !pip install torch transformers datasets scikit-learn --quiet

import random
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModel

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## 1. Get the Data

We load AG News directly from the HuggingFace Hub. It already comes split into `train` and `test`.

In [28]:
raw_dataset = load_dataset("ag_news")
raw_dataset

# save the folder in my current directory
raw_dataset.save_to_disk("ag_news_dataset")


Saving the dataset (0/1 shards):   0%|          | 0/120000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7600 [00:00<?, ? examples/s]

In [29]:


# correct loader for a folder created by save_to_disk()
raw_dataset = load_from_disk("ag_news_dataset")

# Label names (AG News specific - check raw_dataset['train'].features for any dataset)
label_names = raw_dataset["train"].features["label"].names
num_labels = len(label_names)
print("Labels:", label_names)
print("Number of classes:", num_labels)

Labels: ['World', 'Sports', 'Business', 'Sci/Tech']
Number of classes: 4


## 2. Randomly Profile the Data

Before building anything, take a quick look: random samples, text length distribution, and label balance. This catches obvious data issues early (e.g. empty strings, wildly skewed classes).

In [30]:
# Peek at a few random training examples
sample_indices = random.sample(range(len(raw_dataset["train"])), 5)


for i in sample_indices:
    example = raw_dataset["train"][i]
    # print(example)
    print(f"Label: {label_names[example['label']]}")
    print(f"Text  : {example['text'][:200]}...")
    print("-" * 80)

Label: World
Text  : Policeman 'saw fatal train crash' An off-duty policeman watched a train plough into a car on a level crossing  in Berkshire, killing six people....
--------------------------------------------------------------------------------
Label: Sports
Text  : Silver finale for USA In the last event of the 2004 Olympic Games, the United States track team produced one last surprise. Meb Keflezighi, a native of Eritrea who moved to the United States as ...
--------------------------------------------------------------------------------
Label: Sci/Tech
Text  : Compuware Blasts IBM #39;s Legal Tactics Two years ago, IBM was ordered to produce the source code for its products, which Compuware identified as containing its pirated intellectual property. The cod...
--------------------------------------------------------------------------------
Label: World
Text  : Polish Hostage Freed in Iraq Already in Warsaw  WARSAW (Reuters) - A Polish woman kidnapped in Iraq last  month has bee

In [31]:
# Convert a sample to pandas for quick profiling stats
train_df_sample = pd.DataFrame(raw_dataset["train"].shuffle(seed=SEED).select(range(5000)))

print("Label distribution (sample of 5000):")
print(train_df_sample["label"].map(lambda x: label_names[x]).value_counts())

train_df_sample["text_len_words"] = train_df_sample["text"].str.split().apply(len)
print("\nText length (in words) stats:")
print(train_df_sample["text_len_words"].describe())

Label distribution (sample of 5000):
label
Sci/Tech    1324
Sports      1273
World       1253
Business    1150
Name: count, dtype: int64

Text length (in words) stats:
count    5000.000000
mean       37.727600
std         9.794375
min        11.000000
25%        32.000000
50%        37.000000
75%        43.000000
max       144.000000
Name: text_len_words, dtype: float64


## 3. Tokenizer & PyTorch `Dataset`

We use DistilBERT's tokenizer (matches the encoder we'll load later). The `Dataset` class tokenizes text on the fly and returns tensors `input_ids`, `attention_mask`, and `label`.

To keep training fast in this demo, we subsample the full dataset — increase `TRAIN_SUBSET_SIZE` / `TEST_SUBSET_SIZE` (or remove the subsampling) once you're ready for a full run.

In [32]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [33]:
class TextClassificationDataset(Dataset):
    """Wraps text + label lists (or a HF dataset split) into a PyTorch Dataset.
    Tokenizes text lazily inside __getitem__."""

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long),
        }

In [34]:
from sklearn.model_selection import train_test_split

# Pull everything into plain lists first
texts = raw_dataset["train"]["text"]
labels = raw_dataset["train"]["label"]

# --- Step 1: split off the test set first (e.g. 80% temp / 20% test) ---
TEST_SIZE = 0.2   # 20% held out as final test set
VAL_SIZE = 0.1    # 10% of the ORIGINAL data used as validation set (adjust as needed)

temp_texts, test_texts, temp_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=labels,      # remove this arg if labels are continuous / not classification
)

# --- Step 2: split the remaining "temp" data into train / val ---
# VAL_SIZE was defined as a fraction of the ORIGINAL dataset, so we rescale it
# relative to the size of temp_texts (what's left after removing the test set).
val_fraction_of_temp = VAL_SIZE / (1 - TEST_SIZE)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=val_fraction_of_temp,
    random_state=SEED,
    stratify=temp_labels,
)

print("Train size:", len(train_texts))
print("Val size  :", len(val_texts))
print("Test size :", len(test_texts))


Train size: 84000
Val size  : 12000
Test size : 24000


## 4. DataLoader (batch size = 32)

In [35]:
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer, max_len=MAX_LEN)
val_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer, max_len=MAX_LEN)
test_dataset = TextClassificationDataset(test_texts, test_labels, tokenizer, max_len=MAX_LEN)

print("Train size:", len(train_dataset))
print("Val size  :", len(val_dataset))
print("Test size :", len(test_dataset))

# Sanity check a single item
sample = train_dataset[0]
print({k: v.shape for k, v in sample.items()})
print("=" * 80)
print("sample['input_ids']:", sample['input_ids'])
print("sample['attention_mask']:", sample['attention_mask'])
print("sample['label']:", sample['label'])


Train size: 84000
Val size  : 12000
Test size : 24000
{'input_ids': torch.Size([128]), 'attention_mask': torch.Size([128]), 'label': torch.Size([])}
sample['input_ids']: tensor([  101,  4024,  3068,  5176, 19867,  2000,  3627,  2006,  5371,  6631,
         1996,  3185,  1998,  2189,  6088,  2031,  6406,  1037,  9964,  4851,
         1996,  4259,  2457,  2000,  2058, 22299,  1037,  2976,  9023,  2457,
         3247,  2008, 12287, 11153,  1997,  5371,  1011,  6631,  4007,  1012,
          102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 

In [36]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Quick check of one batch
batch = next(iter(train_loader))
print("input_ids shape     :", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("label shape         :", batch["label"].shape)


input_ids shape     : torch.Size([32, 128])
attention_mask shape: torch.Size([32, 128])
label shape         : torch.Size([32])


## 5. Build the Model: Encoder + Fully Connected Head

- **Encoder**: pretrained DistilBERT loaded via `AutoModel` (outputs contextual embeddings).
- **Pooling**: we take the `[CLS]` token's hidden state (first token) as the sentence representation.
- **Head**: a small fully connected network (`Linear -> ReLU -> Dropout -> Linear`) mapping the encoder output to class logits.

The encoder can be fine-tuned (`freeze_encoder=False`) or frozen and used purely as a feature extractor (`freeze_encoder=True`) — both are common; fine-tuning usually gives better accuracy but is slower and needs a smaller learning rate.

In [37]:
class EncoderWithFCClassifier(nn.Module):
    def __init__(self, model_name, num_labels, hidden_dim=256, dropout=0.3, freeze_encoder=False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        encoder_output_dim = self.encoder.config.hidden_size  # 768 for distilbert-base

        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

        self.classifier_head = nn.Sequential(
            nn.Linear(encoder_output_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_labels),
        )

    def forward(self, input_ids, attention_mask):
        encoder_outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # last_hidden_state: (batch, seq_len, hidden_dim)
        cls_embedding = encoder_outputs.last_hidden_state[:, 0, :]  # [CLS] token representation
        logits = self.classifier_head(cls_embedding)
        return logits

In [38]:
model = EncoderWithFCClassifier(
    model_name=MODEL_NAME,
    num_labels=num_labels,
    hidden_dim=256,
    dropout=0.3,
    freeze_encoder=False,  # set True to only train the FC head (faster, lower accuracy ceiling)
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

Total params: 66,560,772
Trainable params: 66,560,772


## 6. Loss Function & Optimizer

In [39]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)  # small LR since we're fine-tuning a pretrained encoder

## 7. Training Loop

In [42]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc="Training", leave=False)
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        # live running stats on the progress bar itself
        progress_bar.set_postfix(loss=loss.item(), acc=correct / total)

    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

In [43]:

# It is basically just a shortcut. Putting @torch.no_grad() at the top of a function does the exact same thing as wrapping the entire inside of the function in a with torch.no_grad(): block.
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc="Evaluating", leave=False)
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        progress_bar.set_postfix(loss=loss.item(), acc=correct / total)

    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

In [44]:
NUM_EPOCHS = 2

history = []  # keep per-epoch metrics if you want to plot them later

for epoch in tqdm(range(1, NUM_EPOCHS + 1), desc="Epochs"):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)   # <- validation, NOT test

    history.append({
        "epoch": epoch,
        "train_loss": train_loss, "train_acc": train_acc,
        "val_loss": val_loss, "val_acc": val_acc,
    })

    print(
        f"Epoch {epoch}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
    )


Epochs:   0%|          | 0/2 [00:00<?, ?it/s]

Training:   0%|          | 0/2625 [00:00<?, ?it/s]

d:\miniconda3\envs\machine_learning\Lib\site-packages\transformers\models\distilbert\modeling_distilbert.py:402: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


KeyboardInterrupt: 

In [ ]:
# visualize the training and validation loss and accuracy over epochs

from matplotlib import pyplot as plt

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot([h["epoch"] for h in history], [h["train_loss"] for h in history], label="Train Loss")
plt.plot([h["epoch"] for h in history], [h["val_loss"] for h in history], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss over Epochs")
plt.legend()


plt.subplot(1,2,2)
plt.plot([h["epoch"]for h in history], [h["train_acc"] for h in history], label="Train Acc")
plt.plot([h["epoch"] for h in history], [h["val_acc"] for h in history], label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy over Epochs")
plt.legend()

plt.show()





## 7b. Final Evaluation on the Held-Out Test Set

Run this **once**, after training/tuning is finished. The test set was never seen during training or used to make decisions about epochs/hyperparameters — that's what `val_loader` was for.


In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Final Test Loss: {test_loss:.4f}, Final Test Acc: {test_acc:.4f}")


## 8. Inference on New Text

In [ ]:
def predict(text, model, tokenizer, label_names, device, max_len=128):
    model.eval()
    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=max_len,
        return_tensors="pt",
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = torch.softmax(logits, dim=1).squeeze(0)
        pred_idx = torch.argmax(probs).item()

    return label_names[pred_idx], probs[pred_idx].item()


example_text = "The national team secured a dramatic victory in the final minutes of the match."
predicted_label, confidence = predict(example_text, model, tokenizer, label_names, device)
print(f"Text: {example_text}")
print(f"Predicted label: {predicted_label} (confidence: {confidence:.4f})")

## 9. Save the Model (optional)

Saves the full model state dict so you can reload it later without retraining.

In [ ]:
torch.save(model.state_dict(), "encoder_fc_classifier.pt")
print("Model saved to encoder_fc_classifier.pt")

# To reload later:
# model = EncoderWithFCClassifier(MODEL_NAME, num_labels)
# model.load_state_dict(torch.load("encoder_fc_classifier.pt", map_location=device))
# model.to(device)